# Notebook 07 — Model Training with SMOTE

Train base learners (Logistic Regression, Gaussian Naive Bayes, Gradient Boosting, Random Forest, XGBoost, LightGBM) on SMOTE-balanced training data using fixed parameters without hyperparameter tuning.


In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


In [2]:
# Load SMOTE-balanced training data and untouched test data
X_train = joblib.load("data/processed/X_train_smote.pkl")
y_train = joblib.load("data/processed/y_train_smote.pkl")

test_df = pd.read_csv("data/processed/test_dataset.csv")
X_test = test_df.drop("Maintenance_Required", axis=1)
y_test = test_df["Maintenance_Required"]

print("SMOTE Training shape:", X_train.shape, y_train.shape)
print("Untouched Test shape:", X_test.shape, y_test.shape)


SMOTE Training shape: (270496, 20) (270496,)
Untouched Test shape: (50000, 20) (50000,)


In [3]:
# Define models including top performers with fixed parameters (NO hyperparameter tuning)
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss", n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=-1, random_state=42, n_jobs=-1, verbosity=-1)
}


In [4]:
# Train base models on SMOTE dataset and evaluate on untouched test data
results = []

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

for name, model in models.items():
    print("=" * 60)
    print(f"Training {name} on SMOTE dataset...")
    print("=" * 60)
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    print(classification_report(y_test, y_pred))
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "ROC-AUC": roc
    })


Training Logistic Regression on SMOTE dataset...


E:\Predictive-vehicle-maintenance-with-road-analysis\Model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       0.94      0.97      0.96     33812
           1       0.94      0.87      0.90     16188

    accuracy                           0.94     50000
   macro avg       0.94      0.92      0.93     50000
weighted avg       0.94      0.94      0.94     50000

Training Gaussian Naive Bayes on SMOTE dataset...
              precision    recall  f1-score   support

           0       0.94      0.97      0.96     33812
           1       0.94      0.87      0.90     16188

    accuracy                           0.94     50000
   macro avg       0.94      0.92      0.93     50000
weighted avg       0.94      0.94      0.94     50000

Training Gradient Boosting on SMOTE dataset...


              precision    recall  f1-score   support

           0       0.94      0.97      0.96     33812
           1       0.94      0.87      0.90     16188

    accuracy                           0.94     50000
   macro avg       0.94      0.92      0.93     50000
weighted avg       0.94      0.94      0.94     50000

Training Random Forest on SMOTE dataset...


              precision    recall  f1-score   support

           0       0.94      0.97      0.96     33812
           1       0.94      0.87      0.90     16188

    accuracy                           0.94     50000
   macro avg       0.94      0.92      0.93     50000
weighted avg       0.94      0.94      0.94     50000

Training XGBoost on SMOTE dataset...


              precision    recall  f1-score   support

           0       0.94      0.97      0.96     33812
           1       0.93      0.87      0.90     16188

    accuracy                           0.94     50000
   macro avg       0.94      0.92      0.93     50000
weighted avg       0.94      0.94      0.94     50000

Training LightGBM on SMOTE dataset...


              precision    recall  f1-score   support

           0       0.94      0.97      0.96     33812
           1       0.93      0.87      0.90     16188

    accuracy                           0.94     50000
   macro avg       0.94      0.92      0.93     50000
weighted avg       0.94      0.94      0.94     50000



In [5]:
# Save all trained base models
filename_map = {
    "Logistic Regression": "logistic_regression_smote.pkl",
    "Gaussian Naive Bayes": "gaussian_naive_bayes_smote.pkl",
    "Gradient Boosting": "gradient_boosting_smote.pkl",
    "Random Forest": "random_forest_smote.pkl",
    "XGBoost": "xgboost_smote.pkl",
    "LightGBM": "lightgbm_smote.pkl"
}

for name, model in models.items():
    fname = filename_map[name]
    joblib.dump(model, f"models/{fname}")
    print(f"Saved model '{name}' to models/{fname}")


Saved model 'Logistic Regression' to models/logistic_regression_smote.pkl
Saved model 'Gaussian Naive Bayes' to models/gaussian_naive_bayes_smote.pkl
Saved model 'Gradient Boosting' to models/gradient_boosting_smote.pkl


Saved model 'Random Forest' to models/random_forest_smote.pkl
Saved model 'XGBoost' to models/xgboost_smote.pkl
Saved model 'LightGBM' to models/lightgbm_smote.pkl


In [6]:
# Save model performance summary
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
results_df.to_csv("results/model_performance_smote.csv", index=False)

print("Base model training results saved successfully:")
results_df


Base model training results saved successfully:


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.94044,0.939864,0.871819,0.904564,0.923146
1,Gaussian Naive Bayes,0.94022,0.942237,0.868606,0.903925,0.923517
2,Gradient Boosting,0.93994,0.938244,0.871880,0.903846,0.924915
3,Random Forest,0.93992,0.937483,0.872622,0.903890,0.924743
4,XGBoost,0.93860,0.933682,0.872313,0.901955,0.924891
5,LightGBM,0.93842,0.933012,0.872436,0.901708,0.924853
